# Dataset de entrenamiento — cross-encoder: ampliación vía research dirigido

Segunda vía para ampliar el corpus, en paralelo al embudo por muestreo aleatorio
(`ds_parte2_corte_const.ipynb`). En vez de samplear al azar sobre las ~21.900
sentencias T/SU del índice completo (rendimiento ~5-7% tras el embudo), parte de una lista
**curada por research dirigido** de sentencias con alta probabilidad temática de ser
pertinentes (`data/candidatas_research.csv`) — el mismo principio que ya funcionó en la parte
1 con las 21 sentencias curadas de redal.org (77% de aciertos vs. ~5-7% del muestreo
aleatorio).

Reusa exactamente el mismo fetch, limpieza de texto, prefiltro de keywords, gate de
pertinencia (Haiku) y extracción completa (Sonnet) ya validados en el notebook del embudo —
la única diferencia es de dónde salen las sentencias candidatas. **Comparte la misma
blacklist** (`dataset_cross_encoder.csv` / `descartados.csv`, el mismo archivo consolidado
que también incluye las sentencias de `ds_parte1_redal.ipynb`) que el embudo, así que ninguno
de los dos notebooks reprocesa lo que el otro ya vio.

**Contenido:**
1. Setup
2. Candidatas del research (excluye ya procesadas por cualquiera de los dos caminos)
3. Descarga y limpieza de texto completo
4. Prefiltro de keywords
5a. Filtro barato de pertinencia (Haiku)
5b. Extracción estructurada completa (Sonnet, solo pertinentes)
6. Construcción de pares (mismo criterio que la parte 1 y el embudo)
7. Loop principal con checkpointing
8. Resultado acumulado

## 1. Setup

In [1]:
# En Colab: descomenta la siguiente línea (o usa `pip install -r requirements.txt` en el venv local)
# !pip install -q beautifulsoup4 requests pandas anthropic python-dotenv

import os
import re
import time
import json

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import anthropic

load_dotenv()  # lee ANTHROPIC_API_KEY del .env en la raíz del repo
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

HEADERS = {"User-Agent": "Mozilla/5.0"}

# Mismas rutas acumuladas que el notebook del embudo — blacklist compartida entre ambos caminos.
# dataset_cross_encoder.csv es el archivo consolidado (redal + corte_const + research).
PARES_PATH = "../../data/dataset_cross_encoder.csv"
DESCARTES_PATH = "../../data/descartados.csv"
CANDIDATAS_RESEARCH_PATH = "../../data/candidatas_research.csv"
COLUMNAS_PARES = ["consulta", "articulo", "tipo", "label", "sentencia_origen"]
COLUMNAS_DESCARTES = ["sentencia", "razon"]

def cargar_si_existe(path, columnas):
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame(columns=columnas)

print("Setup listo.")

Setup listo.


## 2. Candidatas del research (excluye ya procesadas)

`data/candidatas_research.csv` viene de un research dirigido (Claude web) que buscó
específicamente sentencias T/SU de la Corte Constitucional sobre derecho laboral individual
privado, con columnas `sentencia,tema,url`. No toda entrada tiene URL verificada — cuando
falta, se reconstruye con la misma función de la sección 3, así que el campo que realmente
importa es `sentencia`, no `url` (si el research se equivocó en el link exacto no afecta nada,
mientras el número de sentencia sea real).

Se excluyen las que ya aparecen en la blacklist acumulada — sea porque el embudo aleatorio ya
las procesó, o porque una corrida anterior de este mismo notebook ya las intentó.

In [2]:
df_pares_acum = cargar_si_existe(PARES_PATH, COLUMNAS_PARES)
df_descartes_acum = cargar_si_existe(DESCARTES_PATH, COLUMNAS_DESCARTES)
ya_procesadas = set(df_pares_acum["sentencia_origen"]) | set(df_descartes_acum["sentencia"])
print(f"Ya procesadas (embudo + research anterior): {len(ya_procesadas)}")

df_research = pd.read_csv(CANDIDATAS_RESEARCH_PATH)
df_research = df_research.drop_duplicates(subset="sentencia")
print(f"Candidatas del research: {len(df_research)}")

df_muestra = df_research[~df_research["sentencia"].isin(ya_procesadas)].reset_index(drop=True)
df_muestra["sentencia_tipo"] = df_muestra["sentencia"].str.split("-").str[0]
print(f"Candidatas nuevas a intentar: {len(df_muestra)}")
df_muestra["sentencia_tipo"].value_counts()

Ya procesadas (embudo + research anterior): 1495
Candidatas del research: 130
Candidatas nuevas a intentar: 3


sentencia_tipo
T    3
Name: count, dtype: int64

## 3. Descarga y limpieza de texto completo

Idéntico a lo ya validado en el notebook del embudo (mismo patrón de URL, mismo manejo del
shell vacío de la SPA para `SU`, mismo encoding `windows-1252`, mismas anclas de inicio/fin
best-effort). Si el research no trajo una URL verificada, se reconstruye igual a partir del
número de sentencia.

In [3]:
def construir_url_relatoria(tipo, sentencia):
    # sentencia viene como "T-528/17" o "SU-380/21" — separa número y año corto
    numero, anio_corto = sentencia.split("-", 1)[1].split("/")
    anio_largo = ("20" if int(anio_corto) <= 50 else "19") + anio_corto
    sep = "" if tipo == "SU" else "-"
    return f"https://www.corteconstitucional.gov.co/relatoria/{anio_largo}/{tipo}{sep}{numero}-{anio_corto}.htm"

RE_INICIO = re.compile(r"Sentencia\s+(?:No\.?\s*)?(?:T|SU|C)\s*-?\s*\d+\s*/\s*\d+", re.IGNORECASE)
RE_FIN = re.compile(r"comuníquese.{0,80}?cúmplase\.?", re.IGNORECASE)

def descargar_texto_sentencia(url):
    """Descarga y limpia el texto de una sentencia. Devuelve None si falla o si
    la página no trae contenido real (shell de la app Angular del sitio)."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        if b"<app-root" in resp.content:
            return None  # shell vacío de la SPA del sitio, sin contenido real
        soup = BeautifulSoup(resp.content, "html.parser", from_encoding="windows-1252")
        texto = re.sub(r"\s+", " ", soup.get_text(" ")).strip()

        m_inicio = RE_INICIO.search(texto)
        if not m_inicio:
            return None  # no se encontró el título de la sentencia, contenido sospechoso
        texto = texto[m_inicio.start():]

        m_fin = RE_FIN.search(texto)
        if m_fin:
            texto = texto[:m_fin.end()]  # recorta aclaraciones de voto / pie si se encontró el cierre

        return texto
    except Exception:
        return None

## 4. Prefiltro de keywords

Se mantiene como sanity check barato, aunque para candidatas ya curadas por tema se espera que
casi todas lo pasen — si alguna no lo pasa, es señal de que el research se equivocó de tema o
de número de sentencia.

In [4]:
KEYWORDS_ALCANCE = [
    "estabilidad laboral reforzada", "contrato de trabajo", "despido",
    "terminación del contrato", "jornada laboral", "prestaciones sociales",
    "presunción de relación laboral", "liquidación", "recargo nocturno",
    "recargo dominical", "reintegro laboral",
    "relación laboral", "contrato realidad", "justa causa", "indemnización",
    "salario", "cesantías", "vacaciones", "ius variandi", "período de prueba",
    "discriminación laboral", "licencia de maternidad", "fuero de maternidad",
    "proceso disciplinario", "descuento salarial", "acoso laboral",
]

def pasa_prefiltro_keywords(texto):
    texto_lower = texto.lower()
    return any(kw in texto_lower for kw in KEYWORDS_ALCANCE)

## 5a. Filtro barato de pertinencia (Claude Haiku)

Mismo criterio ya corregido tres veces en el notebook del embudo (exclusión incondicional de
pensión/seguridad social en salud, distinción empleado público vs. trabajador oficial,
exclusión de trabajadores independientes y de estudiantes/practicantes, fragmento anclado en
`ANTECEDENTES`). No se repite el historial de correcciones aquí — ver
`ds_parte2_corte_const.ipynb` sección 6a para el detalle de cada caso que motivó cada
ajuste.

In [5]:
PROMPT_GATE = """Lee este fragmento del inicio de una sentencia de la Corte Constitucional colombiana (puede estar incompleto, es solo el comienzo). Con base en el título/tema del caso, decide si es relevante para un dataset de derecho laboral individual colombiano.

Devuelve SOLO un JSON válido, sin texto adicional ni backticks:
{{"pertinente_alcance": true o false, "razon": "una frase breve"}}

pertinente_alcance=true SOLO si el caso trata sobre una relación laboral INDIVIDUAL regida por el Código Sustantivo del Trabajo (o normas equivalentes para trabajadores oficiales): contratación, jornada, terminación del contrato, salario y pagos (constitutivos o no de salario, descuentos), licencias, vacaciones, cesantías (bajo régimen CST/Ley 50 de 1990), traslados (ius variandi), procedimiento disciplinario del empleador, estabilidad laboral reforzada, discriminación laboral en el empleo.

pertinente_alcance=false SIEMPRE, sin excepción, si:
- El tema central es PENSIÓN o SEGURIDAD SOCIAL en cualquier forma: pensión de vejez, invalidez, sobrevivientes, jubilación, sustitución pensional, bono pensional, régimen de prima media o ahorro individual. Esto aplica sin importar qué tan laboral suene el resto del caso — pensión siempre es false.
- El tema central es seguridad social EN SALUD (EPS): pago de licencias (maternidad, incapacidad) reclamado contra una EPS, cobertura o negación de servicios de salud, afiliación al sistema de salud. Es un trámite de seguridad social en salud, no un litigio laboral entre empleador y trabajador — false aunque mencione licencia de maternidad u otro tema nominalmente laboral.
- El accionante es un TRABAJADOR INDEPENDIENTE (por cuenta propia, sin empleador) — el CST no rige relaciones sin empleador, así que no hay relación laboral individual que evaluar. False.
- El accionante es un ESTUDIANTE o PRACTICANTE en formación (práctica académica, convenio docencia-servicio) sin contrato de trabajo real — no hay relación laboral, aunque el conflicto sea con la institución donde hace la práctica. False.
- El empleador es una entidad estatal Y el vínculo es de "empleado público" bajo régimen estatutario/administrativo (docente oficial del magisterio, funcionario de carrera administrativa, personal nombrado, régimen especial propio como el Fondo de Prestaciones Sociales del Magisterio) — NO CST. Si el fundamento del caso son estatutos especiales (ej. Ley 91 de 1989, Estatuto Docente, Decreto 1042 de 1978, normas de carrera administrativa) en vez de CST/Ley 50 de 1990, es false. Excepción: si el vínculo es de "trabajador oficial" (típico de empresas industriales y comerciales del Estado, regido por normas tipo CST o convención colectiva), sí puede ser pertinente.
- Es derecho colectivo (sindicatos, negociación colectiva, huelga, fuero sindical).
- Es función pública en cualquier otro sentido, o el tema es ajeno al laboral individual.
- Lo laboral aparece solo de forma incidental, sin ser el objeto central de la decisión.

Si el fragmento no deja claro el tema o el tipo de vínculo laboral, responde false.

Fragmento:
{texto}
"""

RE_ANTECEDENTES = re.compile(r"ANTECEDENTES")

def filtro_pertinencia_barato(texto, numero, chars_gate=2500):
    m = RE_ANTECEDENTES.search(texto)
    inicio = m.start() if m else 0
    fragmento = texto[inicio:inicio + chars_gate]
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=200,
            messages=[{"role": "user", "content": PROMPT_GATE.format(texto=fragmento)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error en filtro de pertinencia {numero}: {e}")
        return None

## 5b. Extracción estructurada completa (Claude Sonnet, solo pertinentes)

In [6]:
PROMPT_TEMPLATE = """Lee la siguiente sentencia laboral colombiana. Extrae exactamente esta información y devuelve SOLO un JSON válido, sin texto adicional ni backticks:

{{
  "hechos_resumidos": "los hechos del caso en 2-3 líneas, en lenguaje coloquial, como si un trabajador lo contara (ej. 'me despidieron después de...' o 'trabajé X años y...')",
  "pretension": "qué pedía el demandante, en pocas palabras",
  "articulos_fundamento_directo": ["artículos cuya interpretación/aplicación fue DETERMINANTE para resolver el punto concreto en disputa de este caso. Pregunta guía: si se quitara este artículo, ¿cambiaría el razonamiento de por qué se concedió o negó la pretensión específica? Si sí, va aquí. Formato 'CST Art. X' o 'Ley X de YYYY, Art. Y'"],
  "articulos_marco_general": ["artículos que la sentencia cita pero que son principios generales, reglas de interpretación/remisión, o contexto normativo (ej. favorabilidad, analogía, primacía de la realidad, normas constitucionales genéricas) — NO decidieron el punto específico del caso, cualquier sentencia laboral podría citarlos. Mismo formato."],
  "decision": "concedida" o "negada" o "parcial",
  "elemento_no_acreditado": "si la pretensión fue negada o parcial, qué elemento/requisito no se acreditó según el juez; si fue concedida totalmente, deja este campo vacío"
}}

Sentencia:
{texto}
"""

def extraer_estructura(texto, numero, max_chars=15000):
    texto_truncado = texto[:max_chars]
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1200,
            messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(texto=texto_truncado)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error extrayendo {numero}: {e}")
        return None

## 6. Construcción de pares (mismo criterio que la parte 1 y el embudo)

In [7]:
POOL_NEGATIVOS_FACILES = [
    {"articulo": "CST Art. 236", "tema": "licencia de maternidad"},
    {"articulo": "CST Art. 186", "tema": "vacaciones anuales"},
    {"articulo": "CST Art. 161", "tema": "jornada de trabajo"},
    {"articulo": "Ley 100 de 1993, Art. 13", "tema": "seguridad social"},
]

PROMPT_NEGATIVO_DIFICIL = """Dada esta consulta de un caso laboral colombiano:

{consulta}

Y sabiendo que los artículos correctamente aplicables son: {articulos_correctos}

Dame UN artículo real del derecho laboral colombiano (CST, leyes laborales) que esté relacionado temáticamente con la consulta pero que NO sea el fundamento correcto de la decisión — es decir, un artículo que alguien podría confundir con el correcto pero que no aplica aquí.

Devuelve SOLO un JSON válido, sin texto adicional ni backticks:
{{"articulo_incorrecto": "...", "por_que_se_confunde": "..."}}
"""

def generar_negativo_dificil(consulta, articulos_correctos):
    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=300,
            messages=[{"role": "user", "content": PROMPT_NEGATIVO_DIFICIL.format(
                consulta=consulta, articulos_correctos=articulos_correctos)}]
        )
        texto_respuesta = next(b.text for b in response.content if b.type == "text")
        raw = texto_respuesta.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"Error generando negativo difícil: {e}")
        return None

def construir_pares(extraccion, numero):
    consulta = extraccion["hechos_resumidos"]
    articulos_correctos = extraccion.get("articulos_fundamento_directo", [])
    articulos_marco = extraccion.get("articulos_marco_general", [])
    ya_citados = set(articulos_correctos) | set(articulos_marco)

    if not articulos_correctos:
        return None

    pares = [
        {"consulta": consulta, "articulo": art, "tipo": "positivo", "label": 1}
        for art in articulos_correctos
    ]

    candidatos_faciles = [n for n in POOL_NEGATIVOS_FACILES if n["articulo"] not in ya_citados]
    if candidatos_faciles:
        pares.append({
            "consulta": consulta,
            "articulo": candidatos_faciles[0]["articulo"],
            "tipo": "negativo_facil",
            "label": 0,
        })

    negativo_dificil = generar_negativo_dificil(consulta, articulos_correctos)
    if negativo_dificil:
        pares.append({
            "consulta": consulta,
            "articulo": negativo_dificil["articulo_incorrecto"],
            "tipo": "negativo_dificil_placeholder",
            "label": 0,
        })

    df = pd.DataFrame(pares)
    df["sentencia_origen"] = numero
    return df

## 7. Loop principal con checkpointing

Mismo patrón que el embudo: checkpoint acumulado (lo que ya había en `PARES_PATH`/
`DESCARTES_PATH` más lo nuevo de esta corrida) cada `CHECKPOINT_EVERY` sentencias, manejo de
errores por sentencia sin tumbar el loop, pausas entre llamadas.

In [8]:
CHECKPOINT_EVERY = 10

nuevos_pares = []  # lista de DataFrames, uno por sentencia aceptada en ESTA corrida
nuevos_descartes = []  # {"sentencia": ..., "razon": ...}, de ESTA corrida

def guardar_checkpoint():
    partes = [df_pares_acum] + nuevos_pares
    pd.concat(partes, ignore_index=True).to_csv(PARES_PATH, index=False)
    pd.concat([df_descartes_acum, pd.DataFrame(nuevos_descartes, columns=COLUMNAS_DESCARTES)],
              ignore_index=True).to_csv(DESCARTES_PATH, index=False)

for i, row in df_muestra.iterrows():
    numero = row["sentencia"]
    tipo = row["sentencia_tipo"]
    print(f"[{i+1}/{len(df_muestra)}] Procesando {numero}...")

    try:
        url = construir_url_relatoria(tipo, numero)
        texto = descargar_texto_sentencia(url)
        time.sleep(0.5)

        if texto is None:
            nuevos_descartes.append({"sentencia": numero, "razon": "sin_texto_o_url_no_resuelve"})
            continue

        if not pasa_prefiltro_keywords(texto):
            nuevos_descartes.append({"sentencia": numero, "razon": "no_pasa_prefiltro_keywords"})
            continue

        gate = filtro_pertinencia_barato(texto, numero)
        time.sleep(0.5)

        if gate is None:
            nuevos_descartes.append({"sentencia": numero, "razon": "error_filtro_pertinencia"})
            continue

        if not gate.get("pertinente_alcance"):
            nuevos_descartes.append({
                "sentencia": numero,
                "razon": f"llm_no_pertinente: {gate.get('razon', '')}",
            })
            continue

        extraccion = extraer_estructura(texto, numero)
        time.sleep(1)

        if extraccion is None:
            nuevos_descartes.append({"sentencia": numero, "razon": "error_extraccion_llm"})
            continue

        df_par = construir_pares(extraccion, numero)
        time.sleep(1)

        if df_par is not None:
            nuevos_pares.append(df_par)
        else:
            nuevos_descartes.append({"sentencia": numero, "razon": "sin_articulos_fundamento_directo"})

    except Exception as e:
        nuevos_descartes.append({"sentencia": numero, "razon": f"error_inesperado: {e}"})

    if (i + 1) % CHECKPOINT_EVERY == 0:
        guardar_checkpoint()
        print(f"  checkpoint guardado ({i+1}/{len(df_muestra)})")

guardar_checkpoint()
print(f"\nProcesamiento completo. Pares generados de {len(nuevos_pares)} sentencias pertinentes nuevas.")
print(f"Descartadas en esta corrida: {len(nuevos_descartes)} de {len(df_muestra)}")

[1/3] Procesando T-346/25...


[2/3] Procesando T-620/17...


[3/3] Procesando T-438/20...



Procesamiento completo. Pares generados de 1 sentencias pertinentes nuevas.
Descartadas en esta corrida: 2 de 3


## 8. Resultado acumulado

In [9]:
df_dataset_parte2 = pd.concat([df_pares_acum] + nuevos_pares, ignore_index=True) if nuevos_pares else df_pares_acum
df_descartes_total = pd.concat([df_descartes_acum, pd.DataFrame(nuevos_descartes, columns=COLUMNAS_DESCARTES)], ignore_index=True)

print(f"--- Esta corrida ({len(df_muestra)} candidatas del research intentadas) ---")
print(f"Pares nuevos: {sum(len(df) for df in nuevos_pares)} de {len(nuevos_pares)} sentencias pertinentes")

print(f"\n--- Acumulado total (embudo + research) ---")
print(f"Total de pares: {len(df_dataset_parte2)}")
if not df_dataset_parte2.empty:
    print(f"Sentencias representadas: {df_dataset_parte2['sentencia_origen'].nunique()}")
    print(df_dataset_parte2["tipo"].value_counts())

print("\nRazones de descarte (acumulado):")
if not df_descartes_total.empty:
    print(df_descartes_total["razon"].apply(lambda r: r.split(":")[0]).value_counts())

print(f"\nGuardado en {PARES_PATH}")
print(f"Log de descartes en {DESCARTES_PATH}")
df_dataset_parte2.head()

--- Esta corrida (3 candidatas del research intentadas) ---
Pares nuevos: 5 de 1 sentencias pertinentes

--- Acumulado total (embudo + research) ---
Total de pares: 561
Sentencias representadas: 138
tipo
positivo                        285
negativo_facil                  138
negativo_dificil_placeholder    138
Name: count, dtype: int64

Razones de descarte (acumulado):
razon
llm_no_pertinente                   783
no_pasa_prefiltro_keywords          453
sin_articulos_fundamento_directo     51
sin_texto_o_url_no_resuelve          45
revision_manual                      15
sin_texto_o_pdf_no_resuelve          14
no_se_encontro_iframe_pdf             1
Name: count, dtype: int64

Guardado en ../../data/dataset_cross_encoder.csv
Log de descartes en ../../data/descartados.csv


,consulta,articulo,tipo,label,sentencia_origen
0,Trabajé como operador de bus articulado desde ...,CST Art. 127,positivo,1,SL-3630/22
1,Trabajé como operador de bus articulado desde ...,CST Art. 128,positivo,1,SL-3630/22
2,Trabajé como operador de bus articulado desde ...,CST Art. 236,negativo_facil,0,SL-3630/22
3,Trabajé como operador de bus articulado desde ...,CST Art. 130,negativo_dificil_placeholder,0,SL-3630/22
4,Trabajé para el ISS desde septiembre de 2000 h...,"Decreto 2127 de 1945, Art. 20",positivo,1,SL-2858/22
